# Imported Functions

In [ ]:
def read_csv_file_colab(file_name):
  #Reads multiple CSV files from Google Colab's mounted Google Drive.

  from google.colab import drive
  drive.mount('/content/drive')

  # Construct the full file path on Google Drive
  file_path = f"/content/drive/My Drive/Colab Notebooks/SummerResearch2025/{file_name}.csv"

  import pandas as pd

  # Read the CSV file into a DataFrame using pandas
  try:
    df = pd.read_csv(file_path)
  except FileNotFoundError:
    print(f"Error: File '{file_name}' not found on Google Drive.")

  return df

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import pandas as pd
import numpy as np

def dataframe_to_arrays(df, target_column_name, encode=False):
    """
    Converts a DataFrame into feature (X) and target (y) arrays.

    Parameters:
    - df: The input pandas DataFrame.
    - target_column_name: The name of the target column in the DataFrame.
    - encode: Whether to encode non-numerical columns in X and y.

    Returns:
    - X: Features as a NumPy array.
    - y: Target variable as a NumPy array.
    - feature_names: List of feature names.
    - class_names: List of unique class names for the target.
    """
    # Separate features and target
    X = df.drop(columns=[target_column_name])
    y = df[target_column_name]

    feature_names = X.columns.tolist()
    class_names = y.unique().tolist()

    if encode:
        # Encode non-numeric target variable
        if not np.issubdtype(y.dtype, np.number):
            label_encoder = LabelEncoder()
            y = label_encoder.fit_transform(y)
            class_names = label_encoder.classes_.tolist()

        # Encode non-numerical columns in X
        non_numeric_columns = X.select_dtypes(exclude=['number']).columns
        if len(non_numeric_columns) > 0:
            one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
            encoded_features = one_hot_encoder.fit_transform(X[non_numeric_columns])
            encoded_feature_names = one_hot_encoder.get_feature_names_out(non_numeric_columns)

            # Replace non-numerical columns with their encoded versions
            X = pd.DataFrame(np.hstack((X.drop(columns=non_numeric_columns).values, encoded_features)),
                             columns=feature_names[:len(X.columns) - len(non_numeric_columns)] + list(encoded_feature_names))
            feature_names = X.columns.tolist()

    return X.values, y, feature_names, class_names


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, PowerTransformer
from sklearn.model_selection import train_test_split

def scale_dataframes(df, col_to_remove=None, target=None, fraction=None):
    # Remove a specific column from the DataFrame
    if col_to_remove is not None:
      df = df.drop(columns=col_to_remove)

    # Perform stratified sampling if fraction is specified
    if fraction is not None and target is not None:
      df = df.groupby(target, group_keys=False).apply(lambda x: x.sample(frac=fraction))


    # Identify numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    # Identify non-numeric columns
    non_numeric_cols = df.select_dtypes(exclude=['number']).columns

    # Initialize scalers
    std_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler()
    robust_scaler = RobustScaler()
    maxabs_scaler = MaxAbsScaler()
    power_transformer = PowerTransformer()

    # Create a list to store the scaled DataFrames
    scaled_dataframes = []

    # Scale the numeric columns using different scalers and keep the non-numeric columns intact
    for scaler in [std_scaler, minmax_scaler, robust_scaler, maxabs_scaler, power_transformer]:
        scaled_numeric = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)
        scaled_df = pd.concat([scaled_numeric, df[non_numeric_cols].reset_index(drop=True)], axis=1)
        scaled_dataframes.append(scaled_df)

    return scaled_dataframes

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import pandas as pd
import numpy as np

def dataframe_to_arrays(df, target_column_name, encode=False):
    """
    Converts a DataFrame into feature (X) and target (y) arrays.

    Parameters:
    - df: The input pandas DataFrame.
    - target_column_name: The name of the target column in the DataFrame.
    - encode: Whether to encode non-numerical columns in X and y.

    Returns:
    - X: Features as a NumPy array.
    - y: Target variable as a NumPy array.
    - feature_names: List of feature names.
    - class_names: List of unique class names for the target.
    """
    # Separate features and target
    X = df.drop(columns=[target_column_name])
    y = df[target_column_name]

    feature_names = X.columns.tolist()
    class_names = y.unique().tolist()

    if encode:
        # Encode non-numeric target variable
        if not np.issubdtype(y.dtype, np.number):
            label_encoder = LabelEncoder()
            y = label_encoder.fit_transform(y)
            class_names = label_encoder.classes_.tolist()

        # Encode non-numerical columns in X
        non_numeric_columns = X.select_dtypes(exclude=['number']).columns
        if len(non_numeric_columns) > 0:
            one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
            encoded_features = one_hot_encoder.fit_transform(X[non_numeric_columns])
            encoded_feature_names = one_hot_encoder.get_feature_names_out(non_numeric_columns)

            # Replace non-numerical columns with their encoded versions
            X = pd.DataFrame(np.hstack((X.drop(columns=non_numeric_columns).values, encoded_features)),
                             columns=feature_names[:len(X.columns) - len(non_numeric_columns)] + list(encoded_feature_names))
            feature_names = X.columns.tolist()

    return X.values, y, feature_names, class_names

In [ ]:
import numpy as np

def compute_measure(true_label, predicted_label):

  """
  Computes relevant measures that utilize true positive, true negative,
  false positive, and false negative for calculation

  Parameters:
  - true_label: ndarray,contains the true labels of
  a predicted dataset
  - predicted_label: ndarray, contains the predicted labels of
  a predicted dataset

  Returns:
  measurements: ndarray, containing sensitivity, specificity,
  precision, negative predictive value, F1 score, and balanced accuracy
  """
  t_idx = (true_label == predicted_label)  # correctly predicted
  f_idx = np.logical_not(t_idx)  # falsely predicted
  p_idx = (true_label > 0)  # positive targets
  n_idx = np.logical_not(p_idx)  # negative targets
  tp = np.sum(np.logical_and(t_idx, p_idx))  # TP
  tn = np.sum(np.logical_and(t_idx, n_idx))  # TN
  fp = np.sum(n_idx) - tn
  fn = np.sum(p_idx) - tp
  tp_fp_tn_fn_list = [tp, fp, tn, fn]
  tp_fp_tn_fn_list = np.array(tp_fp_tn_fn_list)
  tp = tp_fp_tn_fn_list[0]
  fp = tp_fp_tn_fn_list[1]
  tn = tp_fp_tn_fn_list[2]
  fn = tp_fp_tn_fn_list[3]

  print("TP: ", tp)
  print("FP: ", fp)
  print("TN: ", tn)
  print("FN: ", fn)

  with np.errstate(divide='ignore'):
      sen = (1.0 * tp) / (tp + fn)
      spec = (1.0 * tn) / (tn + fp)
      ppr = (1.0 * tp) / (tp + fp)
      npr = (1.0 * tn) / (tn + fn)
      f1 = tp / (tp + 0.5 * (fp + fn))
      acc = (tp + tn) * 1.0 / (tp + fp + tn + fn)
      balanced_acc = (sen + spec) / 2  # Balanced accuracy calculation
      d = np.log2(1 + acc) + np.log2(1 + (sen + spec) / 2)

  ans = [acc, sen, spec, ppr, npr, f1, d, balanced_acc]
  return ans

import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

def model_metrics(true, pred, proba=None):
    num_classes = len(set(true))
    # calls compute measure
    ans = compute_measure(true, pred)
    print("Accuracy is {0:4f}".format(ans[0]))
    print("Sensitivity is {0:4f}".format(ans[1]))
    print("Specificity is {0:4f}".format(ans[2]))
    print("Precision is {0:4f}".format(ans[3]))
    print("Negative Prediction ratio is {0:3f}".format(ans[4]))
    print("F1-score is {0:3f}".format(ans[5]))
    print("Diagnostic index is {0:4f}".format(ans[6]))
    print("Balanced accuracy is {0:4f}".format(ans[7]))  # Balanced accuracy
    print("\n")
    print(classification_report(true, pred, zero_division=0))
    mic_f1_score = f1_score(true, pred, average='micro')
    mac_f1_score = f1_score(true, pred, average='macro')
    print("\n")
    print('Micro F1-Score:', mic_f1_score)
    print('Macro F1-Score:', mac_f1_score)

    # ROC AUC
    if num_classes == 2 and proba is not None:
        # Check if proba is 1-dimensional
        if proba.ndim == 1:
            roc_auc = roc_auc_score(true, proba)
            fpr, tpr, thresholds = roc_curve(true, proba)
        else:
            roc_auc = roc_auc_score(true, proba[:, 1])
            fpr, tpr, thresholds = roc_curve(true, proba[:, 1])
        print('ROC AUC:', roc_auc)

        # Plot ROC Curve
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend(loc="lower right")
        plt.grid()
        plt.show()

        # Precision-Recall Curve
        precision, recall, pr_thresholds = precision_recall_curve(true, proba if proba.ndim == 1 else proba[:, 1])
        plt.figure(figsize=(8, 6))
        plt.plot(recall, precision, color='green', lw=2, label='Precision-Recall Curve')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curve')
        plt.grid()
        plt.show()
    else:
        print("ROC AUC and Precision-Recall plots are only applicable for binary classification with probabilities.")

    # Confusion Matrix
    cm = confusion_matrix(true, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues', values_format='d')
    plt.title('Confusion Matrix')
    plt.grid(False)
    plt.show()



# Version 1.2

## Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMTPP(nn.Module):
    def __init__(self, num_markers, embed_dim, hidden_dim):
        super().__init__()
        self.num_markers = num_markers
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim

        # Marker embedding layer
        self.embed = nn.Embedding(num_markers, embed_dim)
        self.embed_bias = nn.Parameter(torch.zeros(embed_dim))

        # Hidden state update weights
        self.W_y = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_t = nn.Linear(1, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))

        # Marker generation weights
        self.gen_marker = nn.Linear(hidden_dim, num_markers)

        # Intensity function parameters
        self.v_t = nn.Linear(hidden_dim, 1, bias=False)
        self.w_t = nn.Parameter(torch.ones(1))
        self.b_t = nn.Parameter(torch.zeros(1))

    def forward_event(self, y_j, d_j, h_prev):
        embedded = self.embed(y_j) + self.embed_bias
        d_j = d_j.unsqueeze(1)
        h_j = torch.relu(
            self.W_y(embedded) +
            self.W_t(d_j) +
            self.W_h(h_prev) +
            self.b_h
        )
        return h_j

    def forward_sequence(self, markers, times):
        batch, seq_len = markers.shape
        device = markers.device

        durations = torch.zeros_like(times)
        durations[:, 0] = times[:, 0]
        if seq_len > 1:
            durations[:, 1:] = times[:, 1:] - times[:, :-1]

        h_prev = torch.zeros(batch, self.hidden_dim, device=device)
        hidden_states = []

        for j in range(seq_len):
            h_j = self.forward_event(
                markers[:, j],
                durations[:, j],
                h_prev
            )
            hidden_states.append(h_j)
            h_prev = h_j

        return hidden_states, durations

    def conditional_intensity(self, h_j, t_j, t):
        past_influence = self.v_t(h_j).squeeze(-1)

        if t.dim() > 1:
            delta_t = t - t_j.unsqueeze(-1)
        else:
            delta_t = t - t_j

        mask = (delta_t >= 0).float()
        current_influence = self.w_t * delta_t
        intensity = torch.exp(
            past_influence.unsqueeze(-1) +
                              current_influence +
                              self.b_t
                              )
        return intensity * mask

    def compute_loss(self, markers, times):
        batch, seq_len = markers.shape
        hidden_states, durations = self.forward_sequence(markers, times)
        total_marker_nll = 0
        total_time_nll = 0
        num_events = seq_len - 1

        if num_events < 1:
            return 0, 0, 0

        for j in range(num_events):
            logits = self.gen_marker(hidden_states[j])

            marker_nll = F.cross_entropy(logits,
                                         markers[:, j+1],
                                         reduction='sum')

            total_marker_nll += marker_nll

            d_next = durations[:, j+1]
            A = self.v_t(hidden_states[j]).squeeze(1) + self.b_t
            s = self.w_t * d_next
            exp_A = torch.exp(A)

            if torch.abs(self.w_t) < 1e-10:
                integral = exp_A * d_next
            else:
                integral = exp_A * (torch.exp(s) - 1) / self.w_t

            log_f = A + s - integral
            time_nll = (-log_f).sum()
            total_time_nll += time_nll

        return total_marker_nll, total_time_nll, num_events * batch

    def forward(self, markers, times):
        marker_nll, time_nll, normalizer = self.compute_loss(markers, times)
        if normalizer == 0:
            return torch.tensor(0.0, device=markers.device)
        return (marker_nll + time_nll) / normalizer

    def predict_next_marker(self, h_j):
        logits = self.gen_marker(h_j)
        return F.softmax(logits, dim=-1)

    def predict_next_time(self, h_j, t_j, T=10.0, steps=1000):
        device = h_j.device
        time_grid = torch.linspace(0, T, steps=steps, device=device)
        t_eval = t_j + time_grid

        intensity = self.conditional_intensity(
            h_j,
            torch.tensor([t_j], device=device),
            t_eval.unsqueeze(0)
        ).squeeze()

        # Trapezoidal integration for cumulative intensity
        dt = T / (steps - 1)
        integral = torch.cat([
            torch.zeros(1, device=device),
            torch.cumsum(0.5 * (intensity[:-1] + intensity[1:]) * dt, dim=0)
        ])

        survival = torch.exp(-integral)
        density = intensity * survival

        # Expected time calculation
        t_values = t_eval
        expectation = torch.trapz(t_values * density, t_values)
        return expectation.item()

    def predict_next_event(self, h_j, t_j, strategy='expectation', T=10.0, steps=1000):
        """
        Predict both next marker and next event time.

        Args:
            h_j (torch.Tensor): Current hidden state (batch_size, hidden_dim)
            t_j (torch.Tensor): Current event time (batch_size)
            strategy (str): Prediction strategy -
                'expectation': Predict most probable marker and expected time
                'sample': Sample marker and time from distributions
            T (float): Time horizon for numerical integration
            steps (int): Number of integration steps

        Returns:
            tuple: (next_marker, next_time)
                next_marker: Predicted marker indices (batch_size)
                next_time: Predicted event times (batch_size)
        """

        # Ensure t_j is a tensor of shape [batch_size]
        if not isinstance(t_j, torch.Tensor):
          t_j = torch.full(h_j.size(0), t_j, device=h_j.device)  # Broadcast scalar to batch

        # Predict marker probabilities
        marker_probs = self.predict_next_marker(h_j)  # (batch_size, num_markers)

        # Predict next event time
        batch_size = h_j.size(0)
        next_times = torch.zeros(batch_size, device=h_j.device)

        for i in range(batch_size):
            next_times[i] = self.predict_next_time(
                h_j[i].unsqueeze(0),
                t_j[i].item(),
                T,
                steps
            )

        # Determine marker prediction based on strategy
        if strategy == 'expectation':
            next_markers = torch.argmax(marker_probs, dim=1)
        elif strategy == 'sample':
            next_markers = torch.multinomial(marker_probs, 1).squeeze(1)
        else:
            raise ValueError(f"Unknown strategy: {strategy}. Use 'expectation' or 'sample'")

        return next_markers, next_times

## Psuedo Data

In [ ]:
# Initialize model
model = RMTPP(num_markers=10, embed_dim=32, hidden_dim=64)

# Create sample data
batch_size = 4
seq_len = 5
markers = torch.randint(0, 10, (batch_size, seq_len))
times = torch.cumsum(torch.rand(batch_size, seq_len), dim=1)

# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    optimizer.zero_grad()
    loss = model(markers, times)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item()}")

# Process sequence to get final hidden state
hidden_states, _ = model.forward_sequence(markers, times)
last_hidden = hidden_states[-1]  # [4, 64]
last_times = times[:, -1]  # [4]

# Predict next event (most probable marker + expected time)
next_markers, next_times = model.predict_next_event(
    last_hidden,
    last_times,
    strategy='expectation'
)

print("Predicted markers:", next_markers)
print("Predicted times:", next_times)

# Predict next event (sampled marker + expected time)
next_markers_sampled, next_times_sampled = model.predict_next_event(
    last_hidden,
    last_times,
    strategy='sample'
)

print("Sampled markers:", next_markers_sampled)
print("Sampled times:", next_times_sampled)

Epoch 0, Loss: 2.532050609588623
Epoch 1, Loss: 2.45943284034729
Epoch 2, Loss: 2.3910136222839355
Epoch 3, Loss: 2.325349807739258
Epoch 4, Loss: 2.263413667678833
Epoch 5, Loss: 2.204314947128296
Epoch 6, Loss: 2.147700548171997
Epoch 7, Loss: 2.0932490825653076
Epoch 8, Loss: 2.0410046577453613
Epoch 9, Loss: 1.991085410118103
Predicted markers: tensor([2, 3, 3, 2])
Predicted times: tensor([2.5070, 2.7355, 1.7901, 2.4272])
Sampled markers: tensor([9, 7, 2, 2])
Sampled times: tensor([2.5070, 2.7355, 1.7901, 2.4272])


In [ ]:
markers

tensor([[1, 7, 0, 2, 9],
        [1, 3, 1, 3, 3],
        [8, 4, 3, 8, 5],
        [2, 3, 3, 7, 8]])

In [ ]:
markers.shape

torch.Size([4, 5])

In [ ]:
times

tensor([[0.1573, 0.6653, 0.8169, 1.0760, 2.0348],
        [0.2063, 1.0846, 1.4867, 1.8700, 2.3879],
        [0.2270, 0.4710, 0.9256, 1.1786, 1.2859],
        [0.1198, 0.5146, 0.6269, 1.2919, 2.0670]])

In [ ]:
times.shape

torch.Size([4, 5])

## Credit Data

### Load in Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Load the dataset
file_name = 'creditcard'  # Update path if needed
data = read_csv_file_colab(file_name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [ ]:
# Sort transactions by time (assuming 'Time' column is in seconds)
data = data.sort_values("Time")

# Normalize 'Time' to start at 0 (required for RMTPP)
data["Time"] = data["Time"] - data["Time"].min()

# Convert to PyTorch tensors
markers = torch.tensor(data["Class"].values, dtype=torch.long)  # 0=non-fraud, 1=fraud
times = torch.tensor(data["Time"].values, dtype=torch.float32)   # Timestamps

In [ ]:
times

tensor([0.0000e+00, 0.0000e+00, 1.0000e+00,  ..., 1.7279e+05, 1.7279e+05,
        1.7279e+05])

In [ ]:
markers

tensor([0, 0, 0,  ..., 0, 0, 0])

### Create Sequence Data

In [ ]:
def create_sequences(markers, times, seq_len=60):
    num_sequences = len(markers) // seq_len
    markers = markers[:num_sequences * seq_len]
    times = times[:num_sequences * seq_len]

    # Reshape into (num_sequences, seq_len)
    markers = markers.view(-1, seq_len)
    times = times.view(-1, seq_len)

    return markers, times

seq_len = 60
markers, times = create_sequences(markers, times, seq_len)

In [ ]:
markers

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])

In [ ]:
markers.shape

torch.Size([4746, 60])

In [ ]:
times

tensor([[0.0000e+00, 0.0000e+00, 1.0000e+00,  ..., 3.9000e+01, 4.0000e+01,
         4.1000e+01],
        [4.1000e+01, 4.1000e+01, 4.1000e+01,  ..., 7.6000e+01, 7.6000e+01,
         7.7000e+01],
        [7.7000e+01, 7.7000e+01, 7.8000e+01,  ..., 1.1400e+02, 1.1400e+02,
         1.1500e+02],
        ...,
        [1.7258e+05, 1.7258e+05, 1.7258e+05,  ..., 1.7263e+05, 1.7263e+05,
         1.7263e+05],
        [1.7263e+05, 1.7263e+05, 1.7263e+05,  ..., 1.7269e+05, 1.7269e+05,
         1.7269e+05],
        [1.7270e+05, 1.7270e+05, 1.7270e+05,  ..., 1.7274e+05, 1.7274e+05,
         1.7275e+05]])

In [ ]:
times.shape

torch.Size([4746, 60])

### Create Training Data

In [ ]:
# Split into train/test
markers_train, markers_test, times_train, times_test = train_test_split(
    markers, times, test_size=0.2, random_state=0
)

In [ ]:
markers_train.shape

torch.Size([3796, 60])

In [ ]:
times_train.shape

torch.Size([3796, 60])

In [ ]:
def min_max_scale_times(times):
    """Scale times to [0, 1] range per batch sequence."""
    min_t = times.min(dim=1, keepdim=True).values
    max_t = times.max(dim=1, keepdim=True).values
    scaled_times = (times - min_t) / (max_t - min_t + 1e-8)  # +1e-8 avoids division by zero
    return scaled_times

# Scale the Current Data
times_train_scaled = min_max_scale_times(times_train)
times_test_scaled = min_max_scale_times(times_test)

In [ ]:
times_train_scaled

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.9773, 0.9773, 1.0000],
        [0.0000, 0.0000, 0.0357,  ..., 0.9286, 0.9643, 1.0000],
        [0.0000, 0.0333, 0.1000,  ..., 1.0000, 1.0000, 1.0000],
        ...,
        [0.0000, 0.0385, 0.0385,  ..., 1.0000, 1.0000, 1.0000],
        [0.0000, 0.0217, 0.0543,  ..., 0.9565, 0.9674, 1.0000],
        [0.0000, 0.0000, 0.0256,  ..., 0.9744, 0.9744, 1.0000]])

In [ ]:
times_train_scaled.shape

torch.Size([3796, 60])

In [ ]:
times_test_scaled

tensor([[0.0000, 0.0000, 0.0000,  ..., 1.0000, 1.0000, 1.0000],
        [0.0000, 0.0968, 0.0968,  ..., 0.9032, 0.9677, 1.0000],
        [0.0000, 0.0345, 0.0345,  ..., 0.9310, 0.9655, 1.0000],
        ...,
        [0.0000, 0.0909, 0.0909,  ..., 1.0000, 1.0000, 1.0000],
        [0.0000, 0.0345, 0.0345,  ..., 0.9310, 0.9655, 1.0000],
        [0.0000, 0.0000, 0.0000,  ..., 1.0000, 1.0000, 1.0000]])

In [ ]:
times_test_scaled.shape

torch.Size([950, 60])

### Train the Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model (2 markers: 0=non-fraud, 1=fraud)
model = RMTPP(num_markers=2, embed_dim=32, hidden_dim=64).to(device)

# Convert data to device
markers_train = markers_train.to(device)
times_train_scaled = times_train_scaled.to(device)

# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(4):
    optimizer.zero_grad()
    loss = model(markers_train, times_train_scaled)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.3696679472923279
Epoch 1, Loss: 0.20471206307411194
Epoch 2, Loss: 0.042506515979766846
Epoch 3, Loss: -0.11864130944013596


In [ ]:
batch_size = 4

# Get predictions for a batch
hidden_states, _ = model.forward_sequence(markers_test[:batch_size], times_test_scaled[:batch_size])
last_hidden = hidden_states[-1]  # Shape: [batch_size, hidden_dim]
last_times_scaled = times_test_scaled[:batch_size, -1]  # Shape: [batch_size]

In [ ]:
# Verify Shape
last_hidden.shape

torch.Size([4, 64])

In [ ]:
# Verify Shape
last_times_scaled.shape

torch.Size([4])

In [ ]:
# Predict next event (fraud probability + expected time)
next_markers, next_times_scaled = model.predict_next_event(
    last_hidden,
    last_times_scaled,
    strategy="expectation"
)

print("Predicted fraud probabilities:", next_markers)
print("Predicted next transaction times:", next_times)

Predicted fraud probabilities: tensor([0, 0, 0, 0])
Predicted next transaction times: tensor([2.5070, 2.7355, 1.7901, 2.4272])


### Evaluate Output